<a href="https://colab.research.google.com/github/chaeee01/3DGS-Character-Generation-Pipeline/blob/feature%2Fsam2-preprocess/SAM2_try2_260705.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1️⃣ 첫 번째 셀: 구글 드라이브 마운트 (본계정 연결)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


2️⃣ 두 번째 셀: SAM 2 오픈소스 다운로드 및 환경 설치

In [2]:
# 1. SAM 2 소스코드 다운로드 및 패키지 설치
!git clone https://github.com/facebookresearch/sam2.git
%cd sam2
!pip install -e .

# 2. 가중치(뇌 세포) 파일 저장할 폴더 만들고 다운로드
!mkdir -p checkpoints
!wget -P checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_tiny.pt

# 3. 파이썬이 패키지를 즉시 인식할 수 있도록 시스템 경로 선언
import sys
import os
sys.path.append("/content/sam2")
sys.path.append("/content/sam2/sam2")

print("🎯 SAM 2 환경 설치 및 패키지 로드 완료!")

Cloning into 'sam2'...
remote: Enumerating objects: 1107, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 1107 (delta 10), reused 4 (delta 4), pack-reused 1093 (from 2)
Receiving objects: 100% (1107/1107), 134.85 MiB | 17.03 MiB/s, done.
Resolving deltas: 100% (385/385), done.
/content/sam2
Obtaining file:///content/sam2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 17.6 MB/s eta 0:00:00
  Building editable for SAM-2 (pyproject.toml) ... done
  Created wheel for SAM-2: filename=sam_2-1.0-0.editable-cp312-cp312-linux_x86_64.whl size=13850 sha256=12629a30c2e2bc60e9c8e1ef53665b87a9eb9ea0

3️⃣ 세 번째 셀: 구글 드라이브의 프레임 불러오기 및 예측기(Predictor) 탑재

In [3]:
import torch
from sam2.build_sam import build_sam2_video_predictor

# ⚠️ 아까 1,758장 저장해 둔 구글 드라이브의 폴더 경로
output_dir = "/content/drive/MyDrive/zombie_project/zombie_frames"

# 모델 설정 파일과 다운받은 가중치 매핑
model_cfg = "sam2_hiera_t.yaml"
sam2_checkpoint = "./checkpoints/sam2_hiera_tiny.pt"

# L4 GPU에 예측기 탑재 후 동영상 상태 초기화 (불러오기)
predictor = build_sam2_video_predictor(model_cfg, sam2_checkpoint)
inference_state = predictor.init_state(video_path=output_dir)

print("🎯 [성공] 구글 드라이브에 있던 1758장의 좀비 프레임을 정상적으로 불러와 AI 엔진에 등록했습니다!")

frame loading (JPEG): 100%|██████████| 1758/1758 [04:40<00:00,  6.27it/s]


🎯 [성공] 구글 드라이브에 있던 1758장의 좀비 프레임을 정상적으로 불러와 AI 엔진에 등록했습니다!


4️⃣ 네 번째 셀: 0번 프레임 좌표 지정 (타겟 록온)

In [4]:
import numpy as np

ann_frame_idx = 0  # 첫 번째 프레임
ann_obj_id = 1     # 좀비 오브젝트 ID 번호 (1번)

# 좀비가 있을 법한 가로/세로 픽셀 좌표 설정 (영상 중심부가 기본값)
points = np.array([[500, 500]], dtype=np.float32)
labels = np.array([1], dtype=np.int32) # 1 = 전경(좀비), 0 = 배경

# 첫 프레임에 타겟 점 추가
_, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
    inference_state=inference_state,
    frame_idx=ann_frame_idx,
    obj_id=ann_obj_id,
    points=points,
    labels=labels,
)
print("🎯 0번 프레임 좀비 좌표 타겟팅 완료! 이제 달릴 준비가 되었습니다.")

🎯 0번 프레임 좀비 좌표 타겟팅 완료! 이제 달릴 준비가 되었습니다.


/content/sam2/sam2/sam2_video_predictor.py:786: UserWarning: cannot import name '_C' from 'sam2' (/content/sam2/sam2/__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


좀비 영상 문제 확인. 좌표 타겟팅 코드까지 실행함. 아래 코드 아직 실행 안 함.

### ⚠️ 현재 영상 소스로 돌리면 발생하는 문제점 (실패 원인)
1. 화면 전환(Cut) 시 추적 상실: 영화 쇼츠는 몇 초마다 화면이 탁탁 바뀝니다. SAM 2가 0번 프레임에서 좀비를 잘 잡았더라도, 화면이 바뀌어 다른 사람 표정이 클로즈업되는 순간 AI는 "내가 쫓던 객체가 사라졌다!"라며 추적을 놓치거나 엉뚱한 살색(사람 얼굴)을 좀비로 착각해 누끼를 따버립니다.

2. 떼샷에서의 모호성: 좀비와 사람이 떼로 나오면 SAM 2가 누구를 추적해야 할지 모릅니다. 점 하나([500,500])만 찍어서는 그 수많은 무리 중 정확히 '그 좀비'만 끝까지 골라내기 어렵습니다.

3. 다음 단계(SuGaR, WHAM)에서의 대참사: 이 전처리 영상은 결국 SuGaR(3D 외형 복원)와 WHAM(관절 추적)으로 들어갑니다. 장면이 계속 바뀌거나 다른 사람이 섞여 있으면, WHAM은 관절을 아예 못 잡고 에러가 나며, SuGaR는 좀비가 아니라 기괴하게 일그러진 배경 노이즈 메시를 만들어냅니다.

### 5️⃣ 다섯 번째 셀: 전 프레임 추적 및 투명 배경 PNG 실시간 저장
이 코드는 SAM 2가 마스크를 한 장씩 딸 때마다, 원본 이미지에서 좀비 영역만 쏙 빼내어 배경이 투명한 PNG 시퀀스로 구글 드라이브의 zombie_output_pngs 폴더에 즉시 저장합니다.

### 💡 실행 중 관전 포인트 및 주의사항

1. L4 GPU의 위력: 코드가 실행되면 tqdm 진행바가 뜨면서 초당 수십 장씩 무서운 속도로 숫자가 올라갈 것입니다. 무료 T4와는 차원이 다른 속도를 체감하실 수 있습니다.

2. 중간 확인: 코드가 도는 중에 구글 드라이브의 zombie_project/zombie_output_pngs 폴더를 새로고침해 보세요. 배경이 까맣거나 투명하게 좀비만 덩그러니 남은 이미지 파일들이 실시간으로 쌓이는 걸 눈으로 보실 수 있습니다.

3. 만약 좀비가 아닌 배경이 따진다면?: 만약 4번 셀에서 지정한 [500, 500] 좌표가 좀비 몸통을 빗겨 나가서 배경이 따진다면 진행바를 정지(Interrupt)하시고, 좌표 숫자를 좀비가 있는 정확한 위치로 조금만 수정해서 4번 셀부터 다시 누르면 됩니다.

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# 1. 최종 누끼 이미지들이 저장될 구글 드라이브 폴더 생성
save_dir = "/content/drive/MyDrive/zombie_project/zombie_output_pngs"
os.makedirs(save_dir, exist_ok=True)

# 2. 아까 쪼개두었던 원본 프레임들이 있는 경로
frames_dir = "/content/drive/MyDrive/zombie_project/zombie_frames"
frame_names = sorted([f for f in os.listdir(frames_dir) if f.endswith(".jpg")])

print("🚀 좀비 객체 추적 및 투명 배경 PNG 변환/저장 시작합니다...")

# 3. SAM 2 비디오 추적 루프 가동 (L4 GPU 가속)
# tqdm을 통해 1758장이 실시간으로 처리되는 진척도를 바(Bar)로 보여줍니다.
for out_frame_idx, out_obj_ids, out_mask_logits in tqdm(predictor.propagate_in_video(inference_state), total=len(frame_names)):

    # 현재 프레임의 원본 이미지 읽기
    frame_name = frame_names[out_frame_idx]
    img_path = os.path.join(frames_dir, frame_name)
    src_img = cv2.imread(img_path) # BGR 이미지

    h, w, _ = src_img.shape

    # 기본적으로 완전히 투명한 배경화면(알파채널 0) 생성 (BGRA)
    rgba_output = np.zeros((h, w, 4), dtype=np.uint8)

    for i, out_obj_id in enumerate(out_obj_ids):
        # AI가 찾아낸 좀비 마스크 (True/False 행렬)
        mask = (out_mask_logits[i] > 0.0).cpu().numpy().squeeze()

        # 좀비 영역에 해당하는 픽셀만 원본에서 복사하고, 알파 채널(불투명도)을 255로 설정
        rgba_output[mask, 0:3] = src_img[mask, 0:3] # BGR 복사
        rgba_output[mask, 3] = 255                  # 좀비 영역만 불투명하게

    # 구글 드라이브에 투명 PNG 파일로 저장 (파일명 포맷: zombie_00000.png)
    out_filename = f"zombie_{out_frame_idx:05d}.png"
    cv2.imwrite(os.path.join(save_dir, out_filename), rgba_output)

print(f"\n🎯 [대성공] 총 {len(frame_names)}장의 실사 좀비 누끼 에셋이 구글 드라이브 '{save_dir}' 폴더에 완벽하게 저장되었습니다!")